In [ ]:
# ── MLOps bootstrap (auto-injected by inject_mlops_cell.py) ──────────────────
import os, warnings, mlflow
warnings.filterwarnings("ignore")

SEED = 42
import random, numpy as np
random.seed(SEED)
np.random.seed(SEED)
try:
    import torch; torch.manual_seed(SEED)
except ImportError:
    pass
try:
    import tensorflow as tf; tf.random.set_seed(SEED)
except ImportError:
    pass

_nb_name = os.path.basename(os.path.abspath("__file__") if "__file__" in dir() else ".").replace(".ipynb","")
mlflow.set_tracking_uri("sqlite:///" + str(Path(__file__).parent.parent.parent / "mlflow.db")
                        if "__file__" in dir() else "sqlite:///mlflow.db")
_exp = mlflow.set_experiment(_nb_name or "unnamed_notebook")
print(f"MLflow experiment: {_exp.name}")


# 64 — Filesystem Agent

## Goal
Build an agent that can **search files**, **summarize folders**,
and **rename/organize** content — with **human approval** before changes.

## What You'll Learn
| Concept | Detail |
|---------|--------|
| **Tool gating** | Dangerous actions require human approval |
| **LangGraph StateGraph** | Custom agent flow with approval nodes |
| **File system tools** | List, search, read, and organize files |
| **Human-in-the-loop** | Agent proposes, human approves |

## Architecture
```
User Request → Agent → [Safe tools: list, read, search]
                  ↓
          [Dangerous tools: rename, move, delete]
                  ↓
          Proposal → Human Approval → Execute
```

## Stack
- `LangGraph` StateGraph for approval flow
- `ChatOllama` + custom filesystem tools
- Sandbox directory for safe experimentation

In [ ]:
import os, warnings, shutil, glob, json
os.environ["USE_TF"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
warnings.filterwarnings("ignore")

from pathlib import Path
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

print("All imports OK")

## Step 1 — Create a Sandbox Directory

We work inside a sandboxed folder so the agent can't
accidentally modify real files.

In [ ]:
SANDBOX = Path("./fs_sandbox")
if SANDBOX.exists():
    shutil.rmtree(SANDBOX)
SANDBOX.mkdir(parents=True)

# Create sample files
files = {
    "reports/q1_sales_2024.txt": "Q1 Sales Report\nTotal revenue: $1.2M\nTop product: Widget A\nGrowth: +15%",
    "reports/q2_sales_2024.txt": "Q2 Sales Report\nTotal revenue: $1.5M\nTop product: Premium X\nGrowth: +22%",
    "reports/annual_review_2023.txt": "Annual Review 2023\nTotal employees: 150\nNew hires: 35\nAttrition: 8%",
    "docs/project_plan.md": "# Cloud Migration\n## Timeline: Q1-Q3 2024\n## Budget: $500K\n## Team: 8 engineers",
    "docs/meeting_notes_jan.md": "# Meeting Notes - Jan 15\n- Discussed Q1 goals\n- Budget approved\n- Next meeting: Feb 1",
    "docs/meeting_notes_feb.md": "# Meeting Notes - Feb 1\n- Q1 progress review\n- Hired 3 new engineers\n- Risk: timeline slip",
    "data/customers.csv": "name,email,plan\nAlice,alice@ex.com,pro\nBob,bob@ex.com,basic\n",
    "data/products.csv": "id,name,price\n1,Widget A,29.99\n2,Widget B,49.99\n3,Premium X,129.99",
    "misc/old_draft_v1.txt": "Draft v1 - outdated",
    "misc/old_draft_v2.txt": "Draft v2 - also outdated",
    "misc/todo.txt": "TODO:\n- Clean up old files\n- Update documentation",
    "README.txt": "Project documentation root. See docs/ and reports/ folders.",
}

for path, content in files.items():
    fp = SANDBOX / path
    fp.parent.mkdir(parents=True, exist_ok=True)
    fp.write_text(content)

print(f"Created sandbox with {len(files)} files in {SANDBOX}")

## Step 2 — Define Safe & Gated Tools

**Safe tools**: read-only, can run without approval.  
**Gated tools**: modify files, require human confirmation.

In [ ]:
def _resolve(path: str) -> Path:
    """Resolve path within sandbox, preventing path traversal."""
    resolved = (SANDBOX / path).resolve()
    if not str(resolved).startswith(str(SANDBOX.resolve())):
        raise ValueError("Path traversal blocked!")
    return resolved

# === SAFE TOOLS (read-only) ===

@tool
def list_directory(path: str = ".") -> str:
    """List files and folders in a directory. Use '.' for root sandbox directory."""
    try:
        target = _resolve(path)
        if not target.is_dir():
            return f"{path} is not a directory."
        entries = []
        for item in sorted(target.iterdir()):
            rel = item.relative_to(SANDBOX)
            icon = "📁" if item.is_dir() else "📄"
            size = f" ({item.stat().st_size} bytes)" if item.is_file() else ""
            entries.append(f"  {icon} {rel}{size}")
        return f"Contents of {path}/:\n" + "\n".join(entries)
    except Exception as e:
        return f"Error: {e}"

@tool
def read_file(path: str) -> str:
    """Read the text content of a file."""
    try:
        target = _resolve(path)
        content = target.read_text()
        return f"File: {path}\n{'─'*40}\n{content}"
    except Exception as e:
        return f"Error: {e}"

@tool
def search_files(pattern: str) -> str:
    """Search for files matching a glob pattern (e.g., '**/*.csv' or '**/meeting*')."""
    try:
        matches = list(SANDBOX.glob(pattern))
        if not matches:
            return f"No files matching '{pattern}'."
        results = [f"  {m.relative_to(SANDBOX)}" for m in matches if m.is_file()]
        return f"Found {len(results)} files matching '{pattern}':\n" + "\n".join(results)
    except Exception as e:
        return f"Error: {e}"

@tool
def summarize_folder(path: str = ".") -> str:
    """Get a summary of a folder: file count, total size, file types."""
    try:
        target = _resolve(path)
        if not target.is_dir():
            return f"{path} is not a directory."
        all_files = list(target.rglob("*"))
        files_only = [f for f in all_files if f.is_file()]
        total_size = sum(f.stat().st_size for f in files_only)
        extensions = {}
        for f in files_only:
            ext = f.suffix or "(no extension)"
            extensions[ext] = extensions.get(ext, 0) + 1
        ext_str = ", ".join(f"{ext}: {count}" for ext, count in sorted(extensions.items()))
        return (f"Folder: {path}/\n"
                f"  Files: {len(files_only)}\n"
                f"  Total size: {total_size:,} bytes\n"
                f"  File types: {ext_str}")
    except Exception as e:
        return f"Error: {e}"

# === GATED TOOLS (need approval) ===

@tool
def rename_file(old_path: str, new_path: str) -> str:
    """Rename or move a file. REQUIRES HUMAN APPROVAL. Will ask for confirmation before executing."""
    try:
        src = _resolve(old_path)
        dst = _resolve(new_path)
        if not src.exists():
            return f"Source not found: {old_path}"
        dst.parent.mkdir(parents=True, exist_ok=True)
        src.rename(dst)
        return f"✅ Renamed: {old_path} → {new_path}"
    except Exception as e:
        return f"Error: {e}"

@tool
def delete_file(path: str) -> str:
    """Delete a file. REQUIRES HUMAN APPROVAL. Will ask for confirmation before executing."""
    try:
        target = _resolve(path)
        if not target.exists():
            return f"Not found: {path}"
        if target.is_dir():
            shutil.rmtree(target)
        else:
            target.unlink()
        return f"✅ Deleted: {path}"
    except Exception as e:
        return f"Error: {e}"

safe_tools = [list_directory, read_file, search_files, summarize_folder]
gated_tools = [rename_file, delete_file]
all_tools = safe_tools + gated_tools

print(f"Safe tools: {[t.name for t in safe_tools]}")
print(f"Gated tools: {[t.name for t in gated_tools]}")

## Step 3 — Build Agent with Approval Gate

We use LangGraph's `create_react_agent` with an **interrupt**
mechanism: when the agent wants to call a gated tool,
we intercept, show the proposal, and ask for approval.

In [ ]:
llm = ChatOllama(model="qwen3.5:9b", temperature=0)

FS_SYSTEM_PROMPT = """You are a filesystem assistant that helps organize files.

You can:
- List and search files/folders
- Read file contents
- Summarize folder structures
- Rename, move, or delete files (with user approval)

Rules:
- Always explore before making changes
- Explain what you plan to do before renaming/deleting
- For rename/delete operations, describe the change first and ask for confirmation
- Be careful with destructive operations"""

fs_agent = create_react_agent(
    model=llm,
    tools=all_tools,
    prompt=FS_SYSTEM_PROMPT,
)

def ask_fs_agent(question: str, auto_approve=True):
    """Run the filesystem agent. auto_approve=True for demo purposes."""
    print(f"\n{'='*70}")
    print(f"REQUEST: {question}")
    print(f"{'='*70}")
    result = fs_agent.invoke({"messages": [{"role": "user", "content": question}]})
    for msg in result["messages"]:
        role = msg.__class__.__name__
        if role == "AIMessage" and msg.tool_calls:
            for tc in msg.tool_calls:
                is_gated = tc["name"] in ["rename_file", "delete_file"]
                prefix = "⚠️ GATED" if is_gated else "🔧"
                print(f"\n{prefix} TOOL: {tc['name']}({tc['args']})")
        elif role == "ToolMessage":
            print(f"📋 RESULT: {msg.content[:300]}")
        elif role == "AIMessage" and msg.content:
            print(f"\n🤖 RESPONSE:\n{msg.content}")
    return result

print("Filesystem agent ready!")

In [ ]:
# Task 1: Explore the filesystem
ask_fs_agent("Show me what's in the sandbox. Summarize each folder.")

In [ ]:
# Task 2: Search and read
ask_fs_agent("Find all meeting notes and summarize their key points.")

In [ ]:
# Task 3: Organize (rename) — uses gated tools
ask_fs_agent("The files in misc/ look like drafts. Rename old_draft_v1.txt to archived_draft_v1.txt and old_draft_v2.txt to archived_draft_v2.txt.")

In [ ]:
# Verify the renames happened
print(list_directory.invoke({"path": "misc"}))

## Step 4 — Manual Approval Pattern

In production, you'd use LangGraph's `interrupt()` for real
human-in-the-loop. Here's the conceptual pattern:

In [ ]:
# Conceptual approval pattern (not using actual interrupt for simplicity)
def run_with_approval(question: str):
    """Agent proposes changes, then asks for human approval."""
    # Phase 1: Agent analyzes and proposes
    proposal_prompt = question + "\n\nIMPORTANT: Do NOT execute rename or delete yet. Just describe what changes you would make and why."
    print("📝 Phase 1: Agent proposes changes...")
    result = fs_agent.invoke({"messages": [{"role": "user", "content": proposal_prompt}]})
    
    # Show proposal
    for msg in result["messages"]:
        if msg.__class__.__name__ == "AIMessage" and msg.content:
            print(f"\n📋 PROPOSAL:\n{msg.content}")
    
    # Phase 2: Ask for approval
    print("\n" + "="*50)
    approval = input("Approve these changes? (yes/no): ").strip().lower()
    
    if approval == "yes":
        print("\n✅ Approved! Executing changes...")
        result = fs_agent.invoke({"messages": [{"role": "user", "content": question}]})
        for msg in result["messages"]:
            if msg.__class__.__name__ == "AIMessage" and msg.content:
                print(f"\n🤖 RESULT:\n{msg.content}")
    else:
        print("\n❌ Changes rejected. No modifications made.")

print("Approval pattern defined. (Use run_with_approval() interactively)")

## 🧠 Key Concepts Recap

| Concept | Implementation |
|---------|---------------|
| **Tool gating** | Safe tools (read) vs gated tools (write/delete) |
| **Path traversal guard** | `_resolve()` prevents `../` escapes |
| **Sandbox** | Agent operates only in a designated directory |
| **Human-in-the-loop** | Agent proposes, human approves before execution |
| **LangGraph interrupt** | Production pattern for pausing agent for approval |

## Security Best Practices
```
1. Sandbox the agent — restrict to a specific directory
2. Validate paths — prevent traversal (../../etc/passwd)
3. Gate destructive tools — require approval for rename/delete
4. Audit log — record all tool calls and their results
5. Principle of least privilege — only give tools the agent needs
```